# Easy circuit: saturation regime

Same generator, same model — but the circuit's tap depths are capped at
`CIRC_DEPTH = 5` (uniform over 1..5) instead of 32. From the hard-task
frontier (solved depth ~4 at 7-40M params, depth 5 mid-transition at 50k
steps), a ~17M model trained for 100k steps should come close to saturating
all 256 outputs. Multi-task transfer is also stronger here: ~51 outputs per
depth all pushing the trunk toward the same shallow-gate features.

**Expected ceiling**: a few percent of gate output bits are pure parities of
their light cone (the analysis found stuck 3- and 4-parities on the hard
task); those may stay at chance no matter the budget, capping accuracy
around ~0.99 rather than 1.0. The last cell identifies exactly which outputs
are stuck and checks their Fourier structure.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import os
    %pip -q install -U "jax[cuda12]" optax
    if not os.path.exists("/content/circscale"):
        !git clone https://github.com/amdson/circscale.git /content/circscale
    %cd /content/circscale
    !git pull
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/circscale_runs", exist_ok=True)
    if not os.path.islink("runs"):
        os.symlink("/content/drive/MyDrive/circscale_runs", "runs")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from train import RunConfig, load_run, run

CIRC_DEPTH = 5
STEPS = 100_000
SHAPES = [(128, 4), (360, 7), (512, 8)]   # 0.6M / 7.4M / 17M params
LRS = {(128, 4): 3e-3, (360, 7): 1e-3, (512, 8): 1e-3}  # hard-task tuned
OUT_DIR = "runs/easy"

configs = [
    RunConfig(width=w, mlp_depth=d, lr=LRS[(w, d)], steps=STEPS,
              circ_depth=CIRC_DEPTH, out_dir=OUT_DIR)
    for w, d in SHAPES
]
for cfg in configs:
    print(cfg.name)

## Run (~40 GPU-min total on a T4; resumable)

In [ ]:
for cfg in configs:
    run(cfg)

## Saturation check

In [ ]:
loaded = [(cfg, *load_run(cfg.npz_path)) for cfg in configs if cfg.npz_path.exists()]

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
for cfg, c, d in loaded:
    sel = d["eval_steps"] >= 1000
    D = d["eval_steps"][sel] * c["batch"]
    axes[0].plot(D, d["per_out_loss"][sel].mean(axis=1), lw=1.4,
                 label=f"w{c['width']}d{c['mlp_depth']}")
    depths, acc = d["out_depths"], d["per_out_acc"][-1]
    ks = np.arange(1, CIRC_DEPTH + 1)
    axes[1].plot(ks, [acc[depths == k].mean() for k in ks], "o-",
                 label=f"w{c['width']}d{c['mlp_depth']}")
axes[0].axhline(np.log(2), color="gray", ls=":", lw=1)
axes[0].set(xscale="log", yscale="log", xlabel="samples D", ylabel="eval BCE")
axes[0].legend(fontsize=8); axes[0].set_title("L(D), easy circuit")
axes[1].axhline(1.0, color="gray", ls=":", lw=1)
axes[1].set(xlabel="output tap depth", ylabel="final eval accuracy",
            ylim=(0.45, 1.02))
axes[1].legend(fontsize=8); axes[1].set_title("hardness ladder")
plt.tight_layout()

for cfg, c, d in loaded:
    acc = d["per_out_acc"][-1]
    print(f"w{c['width']}d{c['mlp_depth']}: loss {d['per_out_loss'][-1].mean():.4f}, "
          f"acc {acc.mean():.4f}, outputs below 0.95: {(acc < 0.95).sum()}")

## Who's left? (stuck outputs vs Fourier structure)

For the largest model: every output still under 0.95 accuracy, its tap
depth, and whether it is a pure parity of its input light cone.

In [ ]:
from itertools import combinations

from random_circuit import evaluate_np, sample_circuit

cfg, c, d = loaded[-1]
circuit = sample_circuit(np.random.default_rng(c["circuit_seed"]),
                         c["n_wires"], c["circ_depth"])
depths, acc = d["out_depths"], d["per_out_acc"][-1]
rng = np.random.default_rng(7)
x = rng.integers(0, 2, size=(8192, 256), dtype=np.uint8)
t = 1 - 2.0 * evaluate_np(circuit, x)
s = 1 - 2.0 * x

for w in np.flatnonzero(acc < 0.95):
    base = x[:256]
    y0 = evaluate_np(circuit, base)[:, w]
    deps = [i for i in range(256)
            if (evaluate_np(circuit, base ^ (np.arange(256) == i))[:, w] != y0).any()]
    line = f"wire {w:3d} depth {depths[w]} acc {acc[w]:.3f} cone size {len(deps)}"
    if len(deps) <= 12:
        full = np.mean(t[:, w] * np.prod(s[:, deps], axis=1))
        best_low = max(
            (abs(np.mean(t[:, w] * np.prod(s[:, list(sub)], axis=1)))
             for r in range(1, min(3, len(deps)) + 1)
             for sub in combinations(deps, r)),
            default=0.0,
        )
        line += f" | corr with full-cone XOR {full:+.2f}, best low-degree {best_low:.2f}"
    print(line)

## Notes

- Runs land in `runs/easy/` with `_c256x5` in the name — a separate task,
  never mixed with the depth-32 fits.
- LRs are reused from the hard-task tune; the task change is unlikely to move
  the optimum much, but retune (5k-step grid) if curves look unstable.
- If the 17M model is *not* near-saturating: extend `STEPS` (curves resume),
  or drop `CIRC_DEPTH` to 4. If it saturates too easily, a depth-6 or
  depth-8 cap makes a family of tasks with a movable ceiling — useful for
  placing the saturation point wherever an experiment needs it.